# Step 12 — TF-IDF Text Baseline
### Credit Risk Prediction — Lending Club Dataset

**Goal.** Determine whether borrower text (`emp_title`, `title`, `desc`) contains
predictive signal for the Good vs Bad loan outcome, using a TF-IDF + Logistic
Regression baseline — no PCA, no transformers/embeddings/RAG/GenAI.

**Cohort.** The corrected Step 11 cohort — 41,988 loans (35,573 Good / 6,415 Bad).
As in Step 11, `credit_risk_step6_final.csv` is the sole authoritative source for
the modeling cohort and the `target` column. It is **never** recomputed,
re-derived, or overridden by `loan_status`. `lending-club-loans.csv` is used only
to recover the three text fields via the same exact-match reconciliation built in
Step 11 (Section 0 below reproduces that logic unchanged).

**Sections**
0. Reconstitute the Step 11 cohort (reconciled text on the authoritative target)
1. Combine text fields into one representation per loan
2. Stratified 80/20 train/test split (`random_state=42`)
3. TF-IDF vectorization — fit on training data only
4. Logistic Regression (`class_weight='balanced'`)
5. Evaluation on the untouched test set
6. Top TF-IDF features by class
7. Signal assessment


## 0. Reconstitute the Step 11 cohort

This reproduces Step 11's Section 0 exactly: the authoritative cohort/target
come from `credit_risk_step6_final.csv` and are verified against the known
values before anything else runs; `emp_title`/`title`/`desc` are recovered from
the raw file via the same 13-field exact-match composite key (with the
`emp_length_years` tie-break), never by fuzzy or positional matching. No logic
is changed from Step 11 — this section exists only so Step 12 is runnable
end-to-end on its own.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 100)

S6_PATH = "credit_risk_step6_final.csv"   # authoritative cohort + target
RAW_PATH = "lending-club-loans.csv"       # raw source, text columns only

s6 = pd.read_csv(S6_PATH, low_memory=False)
raw = pd.read_csv(RAW_PATH, low_memory=False, encoding='ISO-8859-1')

print(f"Loaded {S6_PATH}: {s6.shape[0]:,} rows x {s6.shape[1]} columns")
print(f"Loaded {RAW_PATH}: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

EXPECTED_COHORT_SIZE = 41988
EXPECTED_TARGET_COUNTS = {0: 35573, 1: 6415}
assert len(s6) == EXPECTED_COHORT_SIZE, (
    f"credit_risk_step6_final.csv has {len(s6):,} rows, expected {EXPECTED_COHORT_SIZE:,}. "
    "Stopping -- this notebook must not proceed on the wrong cohort."
)
actual_target_counts = s6['target'].value_counts().to_dict()
assert actual_target_counts == EXPECTED_TARGET_COUNTS, (
    f"target distribution is {actual_target_counts}, expected {EXPECTED_TARGET_COUNTS}. "
    "Stopping -- credit_risk_step6_final.csv does not match the authoritative cohort."
)
print()
print("Cohort size and target distribution verified against the authoritative Step 6 values.")

Loaded credit_risk_step6_final.csv: 41,988 rows x 37 columns
Loaded lending-club-loans.csv: 42,538 rows x 117 columns

Cohort size and target distribution verified against the authoritative Step 6 values.


In [2]:
# --- Step 2's Bad definition, applied ONLY to filter which raw rows are
# eligible candidates for matching -- this never overrides s6['target'] ---
BAD_STATUSES = ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']
GOOD_STATUSES = ['Fully Paid', 'Does not meet the credit policy. Status:Fully Paid']

raw_resolved = raw[raw['loan_status'].isin(BAD_STATUSES + GOOD_STATUSES)].copy()

valid_date_or_na = (raw_resolved['earliest_cr_line'].isna() |
                     raw_resolved['earliest_cr_line'].astype(str).str.match(r'^\s*[A-Za-z]{3}-\d{2}\s*$'))
n_corrupted = (~valid_date_or_na).sum()
raw_resolved = raw_resolved[valid_date_or_na].copy()
print(f"Raw candidate rows after Step 2 status filter: {len(raw_resolved) + n_corrupted:,}")
print(f"Dropped for corrupted earliest_cr_line: {n_corrupted}")
print(f"Raw candidate pool for matching: {len(raw_resolved):,}")

Raw candidate rows after Step 2 status filter: 41,989
Dropped for corrupted earliest_cr_line: 1
Raw candidate pool for matching: 41,988


In [3]:
def parse_mon_yy_fix_century(s, pivot_year=2020):
    '''Parse Mon-YY dates, fixing pandas' %y century pivot for dates
    predating 1969 (e.g. Sep-62 -> 1962-09, not 2062-09).'''
    dt = pd.to_datetime(s, format='%b-%y', errors='coerce')
    too_future = dt.dt.year > pivot_year
    return dt.mask(too_future, dt - pd.DateOffset(years=100))

raw_resolved['term_n'] = raw_resolved['term'].str.extract(r'(\d+)').astype(float)
raw_resolved['int_rate_n'] = raw_resolved['int_rate'].str.rstrip('%').astype(float)
raw_resolved['dti_n'] = pd.to_numeric(raw_resolved['dti'], errors='coerce')
raw_resolved['open_acc_n'] = pd.to_numeric(raw_resolved['open_acc'], errors='coerce')
raw_resolved['inq_last_6mths_n'] = pd.to_numeric(raw_resolved['inq_last_6mths'], errors='coerce')
raw_resolved['issue_d_n'] = parse_mon_yy_fix_century(raw_resolved['issue_d']).dt.strftime('%Y-%m-01')
raw_resolved['earliest_cr_line_n'] = parse_mon_yy_fix_century(
    raw_resolved['earliest_cr_line'].astype(str).str.strip()).dt.strftime('%Y-%m-01')

EMP_LEN_MAP = {'< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
               '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8, '9 years': 9,
               '10+ years': 10}
raw_resolved['emp_length_years_n'] = raw_resolved['emp_length'].map(EMP_LEN_MAP)

KEY_COLS_S6 = ['loan_amnt', 'term', 'int_rate', 'installment', 'annual_inc', 'dti',
               'issue_d', 'earliest_cr_line', 'open_acc', 'revol_bal',
               'total_acc', 'delinq_amnt', 'inq_last_6mths']
KEY_COLS_RAW = ['loan_amnt', 'term_n', 'int_rate_n', 'installment', 'annual_inc', 'dti_n',
                'issue_d_n', 'earliest_cr_line_n', 'open_acc_n', 'revol_bal',
                'total_acc', 'delinq_amnt', 'inq_last_6mths_n']
IS_NUMERIC = [True, True, True, True, True, True, False, False, True, True, True, True, True]

def make_key(frame, cols, numeric_flags):
    parts = []
    for c, is_num in zip(cols, numeric_flags):
        v = frame[c]
        if is_num:
            v = pd.to_numeric(v, errors='coerce').astype(float).round(4)
            s = v.astype(str)
            s = s.where(~v.isna(), 'NA')
        else:
            s = v.astype(str)
            s = s.where(~v.isna(), 'NA')
        parts.append(s)
    key = parts[0]
    for p in parts[1:]:
        key = key.str.cat(p, sep='|')
    return key

s6['_key'] = make_key(s6, KEY_COLS_S6, IS_NUMERIC)
raw_resolved['_key'] = make_key(raw_resolved, KEY_COLS_RAW, IS_NUMERIC)

print("Composite key built on both files.")

Composite key built on both files.


In [4]:
# --- Exact-match reconciliation: matched / unmatched / ambiguous ---
raw_groups = raw_resolved.groupby('_key').indices  # key -> array of raw_resolved positions

match_status = []
matched_pos = []
for _, row in s6.iterrows():
    k = row['_key']
    if k not in raw_groups:
        match_status.append('unmatched')
        matched_pos.append(-1)
        continue
    positions = raw_groups[k]
    if len(positions) == 1:
        match_status.append('matched')
        matched_pos.append(positions[0])
        continue
    tie_break = [p for p in positions
                 if raw_resolved.iloc[p]['emp_length_years_n'] == row['emp_length_years']]
    if len(tie_break) == 1:
        match_status.append('matched_tiebreak_emp_length')
        matched_pos.append(tie_break[0])
    else:
        match_status.append('ambiguous')
        matched_pos.append(-1)

s6['_match_status'] = match_status
s6['_raw_pos'] = matched_pos

TEXT_COLS = ['emp_title', 'title', 'desc']
for c in TEXT_COLS:
    recovered = np.where(s6['_raw_pos'].values >= 0,
                          raw_resolved[c].values[np.clip(s6['_raw_pos'].values, 0, None)],
                          np.nan)
    s6[c] = recovered

n_total = len(s6)
n_matched = (s6['_match_status'].isin(['matched', 'matched_tiebreak_emp_length'])).sum()
n_unmatched = (s6['_match_status'] == 'unmatched').sum()
n_ambiguous = (s6['_match_status'] == 'ambiguous').sum()

target_counts = s6['target'].value_counts().to_dict()
RECONCILIATION_OK = (n_total == 41988) and (target_counts == {0: 35573, 1: 6415})
assert RECONCILIATION_OK, "Reconciliation failed cohort/target verification -- stopping before any modeling."

print(f"Matched (text recovered): {n_matched:,} ({n_matched/n_total*100:.2f}%)")
print(f"Unmatched: {n_unmatched:,}   Ambiguous: {n_ambiguous:,}")
print("Reconciliation verified. Proceeding on the corrected 41,988-row cohort.")

df = s6  # authoritative cohort, target untouched, text recovered

Matched (text recovered): 41,984 (99.99%)
Unmatched: 4   Ambiguous: 0
Reconciliation verified. Proceeding on the corrected 41,988-row cohort.


## 1. Combine text fields into one representation per loan

`emp_title`, `title`, and `desc` are concatenated into a single `combined_text`
field per loan. Missing text in any of the three fields is filled with an empty
string before concatenation (not dropped, not imputed with placeholder words) —
so a loan with, say, only a `title` still contributes that text rather than
being excluded.

In [5]:
for c in TEXT_COLS:
    df[c] = df[c].fillna('')

df['combined_text'] = (df['emp_title'] + ' ' + df['title'] + ' ' + df['desc']).str.strip()

n_empty_combined = (df['combined_text'] == '').sum()
print(f"Rows with completely empty combined text (all 3 fields missing): {n_empty_combined:,} "
      f"({n_empty_combined/len(df)*100:.2f}%)")
df[['emp_title', 'title', 'desc', 'combined_text']].head(3)

Rows with completely empty combined text (all 3 fields missing): 5 (0.01%)


,emp_title,title,desc,combined_text
0,,Computer,Borrower added on 12/22/11 > I need to upgra...,Computer Borrower added on 12/22/11 > I need...
1,Ryder,bike,Borrower added on 12/22/11 > I plan to use t...,Ryder bike Borrower added on 12/22/11 > I pl...
2,,real estate business,,real estate business


## 2. Stratified train/test split

Same split methodology as the rest of the project: **stratified 80/20**,
`random_state=42`. This split is independent of Steps 7–10's structured-feature
split (different feature space, same cohort and same splitting convention) and
is frozen before any vectorization happens.

In [6]:
from sklearn.model_selection import train_test_split

X_text = df['combined_text']
y = df['target']

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.20, stratify=y, random_state=42
)

print(f"Train: {len(X_train_text):,} rows   Test: {len(X_test_text):,} rows")
print(f"Train target distribution: {y_train.value_counts().to_dict()}")
print(f"Test target distribution:  {y_test.value_counts().to_dict()}")

Train: 33,590 rows   Test: 8,398 rows
Train target distribution: {0: 28458, 1: 5132}
Test target distribution:  {0: 7115, 1: 1283}


## 3. TF-IDF vectorization — fit on training data only

The vectorizer is fit **only** on `X_train_text` and then used to `transform`
(never `fit_transform`) the test text, so no information from the test set
leaks into the vocabulary or IDF weights.

Configuration: `lowercase=True`, `stop_words='english'`, `ngram_range=(1,2)`,
`min_df=5`, `max_df=0.95`.

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.95
)

X_train_tfidf = vectorizer.fit_transform(X_train_text)   # fit + transform on TRAIN only
X_test_tfidf = vectorizer.transform(X_test_text)         # transform only, using train's vocabulary/IDF

n_features = len(vectorizer.get_feature_names_out())
print(f"TF-IDF vocabulary size (features): {n_features:,}")
print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape:  {X_test_tfidf.shape}")

TF-IDF vocabulary size (features): 29,988
Train matrix shape: (33590, 29988)
Test matrix shape:  (8398, 29988)


## 4. Logistic Regression on the TF-IDF representation

`class_weight='balanced'` is used to handle the ~5.5:1 Good:Bad imbalance
directly in the loss, rather than resampling (e.g. SMOTE), which is
inappropriate on high-dimensional sparse TF-IDF features. `liblinear` is used
as the solver, well suited to high-dimensional sparse input.

In [8]:
from sklearn.linear_model import LogisticRegression

LOGREG_CONFIG = dict(
    class_weight='balanced',
    solver='liblinear',
    max_iter=1000,
    random_state=42
)

clf = LogisticRegression(**LOGREG_CONFIG)
clf.fit(X_train_tfidf, y_train)

print("Logistic Regression configuration used:")
for k, v in LOGREG_CONFIG.items():
    print(f"  {k} = {v}")

Logistic Regression configuration used:
  class_weight = balanced
  solver = liblinear
  max_iter = 1000
  random_state = 42


## 5. Evaluation on the untouched test set

All metrics below are computed **only** on `X_test_tfidf` / `y_test` — the
20% held out before vectorization and never used for fitting the vectorizer or
the classifier.

In [9]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score, confusion_matrix,
                              classification_report)

y_pred = clf.predict(X_test_tfidf)
y_proba = clf.predict_proba(X_test_tfidf)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
cm = confusion_matrix(y_test, y_pred)

print("=== Test set metrics (Bad = positive class) ===")
print(f"Accuracy:            {acc:.4f}")
print(f"Precision (Bad):     {prec:.4f}")
print(f"Recall (Bad):        {rec:.4f}")
print(f"F1-score (Bad):      {f1:.4f}")
print(f"ROC-AUC:             {roc_auc:.4f}")
print(f"PR-AUC (Avg Prec.):  {pr_auc:.4f}")

=== Test set metrics (Bad = positive class) ===
Accuracy:            0.6744
Precision (Bad):     0.2040
Recall (Bad):        0.3897
F1-score (Bad):      0.2678
ROC-AUC:             0.6058
PR-AUC (Avg Prec.):  0.2164


In [10]:
cm_df = pd.DataFrame(cm,
                      index=['Actual: Good (0)', 'Actual: Bad (1)'],
                      columns=['Pred: Good (0)', 'Pred: Bad (1)'])
print("Confusion matrix:")
display(cm_df)

print()
print(classification_report(y_test, y_pred, target_names=['Good (0)', 'Bad (1)']))

Confusion matrix:


,Pred: Good (0),Pred: Bad (1)
Actual: Good (0),5164,1951
Actual: Bad (1),783,500



              precision    recall  f1-score   support

    Good (0)       0.87      0.73      0.79      7115
     Bad (1)       0.20      0.39      0.27      1283

    accuracy                           0.67      8398
   macro avg       0.54      0.56      0.53      8398
weighted avg       0.77      0.67      0.71      8398



## 6. Top TF-IDF features by class

Logistic Regression coefficients on the TF-IDF features. A **positive**
coefficient means the feature pushes the model's prediction toward **Bad**;
a **negative** coefficient means it pushes toward **Good**. These describe
what the model associates with each class — they are not evidence that any
individual word causes default.

In [11]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = clf.coef_[0]

top_pos_idx = np.argsort(coefs)[-20:][::-1]   # most positive -> associated with Bad
top_neg_idx = np.argsort(coefs)[:20]          # most negative -> associated with Good

top_bad_df = pd.DataFrame({
    'feature': feature_names[top_pos_idx],
    'coefficient': coefs[top_pos_idx]
})
top_good_df = pd.DataFrame({
    'feature': feature_names[top_neg_idx],
    'coefficient': coefs[top_neg_idx]
})

print("Top 20 TF-IDF features associated with Bad loan predictions (most positive coefficients):")
display(top_bad_df)

print()
print("Top 20 TF-IDF features associated with Good loan predictions (most negative coefficients):")
display(top_good_df)

Top 20 TF-IDF features associated with Bad loan predictions (most positive coefficients):


,feature,coefficient
0,11 job,2.178837
1,00,2.106011
2,group home,2.080754
3,business,2.029271
4,bills,2.022926
5,county credit,1.964193
6,casting,1.922502
7,homes,1.857591
8,need,1.853046
9,medical expenses,1.850071



Top 20 TF-IDF features associated with Good loan predictions (most negative coefficients):


,feature,coefficient
0,rate,-2.294772
1,ring,-2.238215
2,land,-2.146498
3,apr,-2.101365
4,salary,-2.093145
5,pay minimum,-2.072101
6,pool,-2.034894
7,late payment,-1.962664
8,research,-1.932995
9,engagement,-1.924350


## 7. Signal assessment

The cell below builds this section's conclusion programmatically from the
metrics computed above, so it reflects whatever this run actually measured.

In [12]:
from IPython.display import Markdown, display as ipy_display

MAJORITY_BASELINE_ACC = (y_test == 0).mean()

lines = []
lines.append("### Does text appear to contain useful predictive signal?")
lines.append("")
lines.append(f"- **ROC-AUC = {roc_auc:.4f}** — meaningfully above the 0.50 no-signal baseline, "
              "so the TF-IDF text representation does carry *some* separable information "
              "between Good and Bad loans; it is a weak-to-modest signal, not a strong one.")
lines.append(f"- **PR-AUC (Average Precision) = {pr_auc:.4f}** vs. the Bad-class base rate of "
              f"{y_test.mean():.4f} in the test set — PR-AUC exceeds the base rate, which is the "
              "relevant comparison under this class imbalance (ROC-AUC alone can look better than "
              "the model is at identifying the minority class).")
lines.append(f"- **Accuracy = {acc:.4f}** is *below* the majority-class baseline "
              f"({MAJORITY_BASELINE_ACC:.4f} from always predicting Good) — expected and acceptable "
              "here, since `class_weight='balanced'` deliberately trades accuracy for better recall "
              "on the minority (Bad) class rather than defaulting to the majority class.")
lines.append(f"- **Recall (Bad) = {rec:.4f}, Precision (Bad) = {rec and prec:.4f}** — the model "
              "recovers a meaningful fraction of Bad loans from text alone but with substantial "
              "false positives, consistent with text being a supplementary rather than primary "
              "signal.")
lines.append(f"- The top associated features (Section 6) include plausible financial-hardship and "
              "purpose-related terms (e.g. medical/bill-related language skewing toward Bad; "
              "rate/APR/consolidation language skewing toward Good) alongside some tokens that look "
              "like residual formatting/date artifacts from the `desc` field's boilerplate "
              "(\"Borrower added on MM/DD/YY ...\") rather than semantic content — a reminder that "
              "these are model associations, not confirmed causal or even fully \"clean\" semantic "
              "signals.")
lines.append("")
lines.append(f"**Conclusion:** on this cohort, TF-IDF text carries a real but modest amount of "
              f"predictive signal for Good vs. Bad (ROC-AUC {roc_auc:.3f}, PR-AUC {pr_auc:.3f} vs. "
              f"a {y_test.mean():.3f} base rate) — clearly better than chance, well short of a strong "
              "standalone classifier. This does not yet say whether it adds anything on top of the "
              "structured features from Steps 7–10; that comparison is future work.")
lines.append("")
lines.append("**Caveat:** this is a single baseline configuration (fixed TF-IDF hyperparameters, "
              "no tuning). It establishes whether *any* signal exists, not the ceiling on what a "
              "text model could achieve.")

summary_md = "\n".join(lines)
ipy_display(Markdown(summary_md))

### Does text appear to contain useful predictive signal?

- **ROC-AUC = 0.6058** — meaningfully above the 0.50 no-signal baseline, so the TF-IDF text representation does carry *some* separable information between Good and Bad loans; it is a weak-to-modest signal, not a strong one.
- **PR-AUC (Average Precision) = 0.2164** vs. the Bad-class base rate of 0.1528 in the test set — PR-AUC exceeds the base rate, which is the relevant comparison under this class imbalance (ROC-AUC alone can look better than the model is at identifying the minority class).
- **Accuracy = 0.6744** is *below* the majority-class baseline (0.8472 from always predicting Good) — expected and acceptable here, since `class_weight='balanced'` deliberately trades accuracy for better recall on the minority (Bad) class rather than defaulting to the majority class.
- **Recall (Bad) = 0.3897, Precision (Bad) = 0.2040** — the model recovers a meaningful fraction of Bad loans from text alone but with substantial false positives, consistent with text being a supplementary rather than primary signal.
- The top associated features (Section 6) include plausible financial-hardship and purpose-related terms (e.g. medical/bill-related language skewing toward Bad; rate/APR/consolidation language skewing toward Good) alongside some tokens that look like residual formatting/date artifacts from the `desc` field's boilerplate ("Borrower added on MM/DD/YY ...") rather than semantic content — a reminder that these are model associations, not confirmed causal or even fully "clean" semantic signals.

**Conclusion:** on this cohort, TF-IDF text carries a real but modest amount of predictive signal for Good vs. Bad (ROC-AUC 0.606, PR-AUC 0.216 vs. a 0.153 base rate) — clearly better than chance, well short of a strong standalone classifier. This does not yet say whether it adds anything on top of the structured features from Steps 7–10; that comparison is future work.

**Caveat:** this is a single baseline configuration (fixed TF-IDF hyperparameters, no tuning). It establishes whether *any* signal exists, not the ceiling on what a text model could achieve.

## Notebook scope recap

- **`credit_risk_step6_final.csv` remains the sole authoritative source for the
  cohort and `target`** — 41,988 rows, 35,573 Good / 6,415 Bad, never
  recomputed or overridden by `loan_status`.
- Text was combined from `emp_title` + `title` + `desc` with missing values
  filled as empty strings (never dropped).
- Split: stratified 80/20, `random_state=42`, consistent with the rest of the
  project.
- TF-IDF was fit **only** on the training text; the test text was only
  `transform`-ed.
- Class imbalance was handled via `class_weight='balanced'` on the classifier,
  not SMOTE (inappropriate for sparse TF-IDF features).
- No PCA, no transformers/embeddings/RAG/GenAI.
- Evaluation is on the untouched test set only.
- No comparison to the structured model, no combined model, no SHAP yet —
  scoped separately for a later step.
